# RL Teaching — Day 2
**CartPole · Maze: Reward Design Comparison**

> 📌 **Before you start:** Go to **File → Save a copy in Drive** to make your own editable copy.

Focus on **what changes when you adjust the 🔧 parameters** and re-run the cell.

In [ ]:
# ── Setup ────────────────────────────────────────────────────
!pip install gymnasium matplotlib numpy --quiet
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
print('✅ Setup complete!')

---
## Part 1 · CartPole — Continuous State Space

CartPole: keep a pole balanced on a moving cart.
The state has **4 continuous values** — not grid squares like the maze.

Q-learning needs *discrete* states, so we divide each dimension into **bins**.
Rein Room does this automatically; here you can see exactly what happens.

```
State dimension    | Range          | Bins
Cart position      | -2.4  to  2.4  | 10
Cart velocity      | -3.0  to  3.0  | 10
Pole angle         | -0.25 to  0.25 | 10
Pole ang. velocity | -3.0  to  3.0  | 10
Total possible states: 10 x 10 x 10 x 10 = 10,000
```

### 🔧 Your task
Watch the reward curve. When does improvement start?
Does it ever drop suddenly after improving? Why might that happen?

In [ ]:
# ════════════════════════════════════════════
alpha      = 0.5   # 🔧 learning rate    — try: 0.2 / 0.5 / 0.8
gamma      = 0.99  # 🔧 discount factor  — try: 0.9 / 0.99
epsilon    = 0.3   # 🔧 exploration rate — try: 0.1 / 0.3 / 0.5
n_episodes = 500
# ════════════════════════════════════════════

env    = gym.make('CartPole-v1')
n_bins = 10
bins   = [
    np.linspace(-2.4,  2.4,  n_bins),
    np.linspace(-3.0,  3.0,  n_bins),
    np.linspace(-0.25, 0.25, n_bins),
    np.linspace(-3.0,  3.0,  n_bins),
]

def discretize(obs):
    return tuple(min(np.digitize(obs[i], bins[i]), n_bins-1) for i in range(4))

Q      = {}
get_q  = lambda s, a: Q.get((s, a), 0.0)
best_a = lambda s: max(range(2), key=lambda a: get_q(s, a))
episode_rewards, episode_lengths = [], []

for ep in range(n_episodes):
    obs, _  = env.reset()
    state   = discretize(obs)
    total, steps, done = 0, 0, False
    while not done:
        action = env.action_space.sample() if np.random.random() < epsilon else best_a(state)
        nobs, r, ter, tru, _ = env.step(action)
        done   = ter or tru
        ns     = discretize(nobs)
        old_q  = get_q(state, action)
        nmax   = max(get_q(ns, a) for a in range(2))
        Q[(state, action)] = old_q + alpha * (r + gamma * nmax - old_q)
        state, total, steps = ns, total + r, steps + 1
    episode_rewards.append(total)
    episode_lengths.append(steps)

window = 30
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(np.convolve(episode_rewards, np.ones(window)/window, mode='valid'), lw=2)
ax1.axhline(195, color='red', linestyle='--', label='Solved threshold (195)')
ax1.set(title=f'CartPole Reward | alpha={alpha} gamma={gamma} epsilon={epsilon}',
        xlabel='Episode', ylabel='Total Reward (max 200)')
ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(np.convolve(episode_lengths, np.ones(window)/window, mode='valid'), color='orange', lw=2)
ax2.set(title='Episode Length  (longer = better balance)',
        xlabel='Episode', ylabel='Steps')
ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f"Q-table entries used : {len(Q):,} / {10**4*2:,} possible")
print(f"Avg reward (last 50) : {np.mean(episode_rewards[-50:]):.1f} / 200")

---
## Part 2 · Maze — Reward Design: Sparse vs. Shaped

Same 5×5 maze as Day 1. We train the agent **twice** with different reward functions.

| Version | Reach goal | Each step | Hit wall |
|---------|-----------|-----------|----------|
| **Sparse** (standard) | +1.0 | 0 | 0 |
| **Shaped** (Rein Room style) | +1.0 | -0.01 | -0.05 |

**Key question:** Why does reward design matter?
- Sparse reward: the agent gets no feedback until it accidentally reaches the goal.
  It may wander for hundreds of episodes before learning anything.
- Shaped reward: the agent gets a small penalty each step, so it learns to
  take shorter paths — feedback is immediate and continuous.

### 🔧 Your task
Run the cell and compare the two learning curves.
- Which one starts improving earlier?
- Which one finds a shorter path (fewer steps)?
- Why does immediate feedback make learning easier?

In [ ]:
# Fixed parameters — focus on comparing reward functions
alpha, gamma, epsilon, n_episodes = 0.5, 0.95, 0.2, 1000
window = 50

ROWS, COLS = 5, 5
WALLS = {(1,1), (1,3), (3,1), (3,3)}
START, GOAL = (0,0), (4,4)
ACTIONS = [(-1,0),(1,0),(0,-1),(0,1)]
N_STATES, N_ACTIONS = ROWS * COLS, 4

def state_id(r, c): return r * COLS + c

def step_maze(r, c, a, shaped):
    nr, nc = r + ACTIONS[a][0], c + ACTIONS[a][1]
    if nr < 0 or nr >= ROWS or nc < 0 or nc >= COLS or (nr,nc) in WALLS:
        reward = -0.05 if shaped else 0.0
        return r, c, reward, False
    if (nr, nc) == GOAL:
        return nr, nc, 1.0, True
    reward = -0.01 if shaped else 0.0
    return nr, nc, reward, False

def run_maze(shaped):
    Q = np.zeros((N_STATES, N_ACTIONS))
    ep_rewards, ep_lengths = [], []
    for _ in range(n_episodes):
        r, c = START
        total, steps, done = 0.0, 0, False
        while not done and steps < 300:
            s = state_id(r, c)
            a = np.random.randint(N_ACTIONS) if np.random.random() < epsilon else np.argmax(Q[s])
            nr, nc, reward, done = step_maze(r, c, a, shaped)
            ns = state_id(nr, nc)
            Q[s, a] += alpha * (reward + gamma * np.max(Q[ns]) - Q[s, a])
            r, c, total, steps = nr, nc, total + reward, steps + 1
        ep_rewards.append(total)
        ep_lengths.append(steps)
    return ep_rewards, ep_lengths

print('Training... (~5 seconds)')
r_sparse, l_sparse = run_maze(shaped=False)
r_shaped, l_shaped = run_maze(shaped=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

for rewards, label, color in [
    (r_sparse, 'Sparse reward (no step penalty)',   'coral'),
    (r_shaped, 'Shaped reward (step penalty -0.01)', 'steelblue'),
]:
    s = np.convolve(rewards, np.ones(window)/window, mode='valid')
    ax1.plot(s, label=label, color=color, lw=2)
ax1.set(title='Episode Reward: Sparse vs. Shaped',
        xlabel='Episode', ylabel='Total Reward')
ax1.legend(); ax1.grid(alpha=0.3)

for lengths, label, color in [
    (l_sparse, 'Sparse reward', 'coral'),
    (l_shaped, 'Shaped reward', 'steelblue'),
]:
    s = np.convolve(lengths, np.ones(window)/window, mode='valid')
    ax2.plot(s, label=label, color=color, lw=2)
ax2.set(title='Episode Length (fewer steps = better path)',
        xlabel='Episode', ylabel='Steps')
ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

sparse_success = sum(1 for r in r_sparse[-200:] if r > 0)
shaped_success = sum(1 for r in r_shaped[-200:] if r > 0)
print(f"Sparse reward — success rate (last 200): {sparse_success/200:.1%}")
print(f"Shaped reward — success rate (last 200): {shaped_success/200:.1%}")